# Ausgrid monthly VAE · validation diagnostics and decoder ablation

This notebook follows `Ausgrid_Monthly_Conditional_VAE.ipynb`. It checks the distribution errors, retrains the **same encoder and loss with separate solar/non-solar decoder heads**, and fits a simple annual-total correction. The original joint VAE checkpoints are loaded for comparison.

**Input:** `monthly_work_package_1/` and `outputs/monthly_vae/` beneath the original `ausgrid-synthetic` project. In Colab, the notebook looks in both `MyDrive/ausgrid-synthetic/` and `MyDrive/ausgrid-synthetic/ausgrid-synthetic/`. **Output:** `outputs/monthly_vae_ablation/`.

Only train and validation household trajectories are loaded. The 974 validation households are deterministically divided into 487 calibration and 487 evaluation households, stratified by solar status. Neither this experiment nor its diagnostics accesses the prior test-household trajectories. The full validation set was previously used for early stopping of the shared model; this remains an *exploratory validation comparison*, not a new independent final test.


In [ ]:
from __future__ import annotations
import hashlib
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from scipy.stats import spearmanr, wasserstein_distance
from sklearn.metrics import roc_auc_score

YEARS = (2012, 2013, 2014)
SEEDS = (11, 23, 37)
CHANNELS = ("base_kwh", "controlled_kwh", "generation_kwh")

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


## 1. Load the same prepared panel and fixed split

Only training and validation trajectories enter memory. Confirm the existing Phase 2 checkpoints were trained with the exact same panel, household split, LGA vocabulary, and scaler.


In [ ]:
def resolve_project_root() -> Path:
    """Find prepared output locally or in one of the two prior Colab layouts."""
    workdir = Path.cwd().resolve()
    drive_root = Path("/content/drive/MyDrive/ausgrid-synthetic")
    candidates = [workdir, workdir.parent, workdir / "ausgrid-synthetic",
                  drive_root, drive_root / "ausgrid-synthetic"]

    def ready(p: Path) -> bool:
        parent = p / "monthly_work_package_1"
        return all((parent / f).is_file() for f in (
            "monthly_panel_2007_2014.csv.gz", "household_splits.csv", "baseline_metrics.csv"))

    root = next((p for p in candidates if ready(p)), None)
    if root is None:
        try:
            from google.colab import drive
        except ImportError as exc:
            raise FileNotFoundError("Run the first monthly notebook in ausgrid-synthetic before this one.") from exc
        drive.mount("/content/drive")
        root = next((p for p in (drive_root, drive_root / "ausgrid-synthetic") if ready(p)), None)
    if root is None:
        raise FileNotFoundError("First monthly notebook outputs not found in either MyDrive project layout.")
    return root.resolve()


def load_monthly_data(prepared_dir: Path, include_test: bool = True) -> tuple[dict, dict, pd.DataFrame]:
    panel = pd.read_csv(prepared_dir / "monthly_panel_2007_2014.csv.gz", parse_dates=["month"])
    sites = pd.read_csv(prepared_dir / "household_splits.csv")
    baseline = pd.read_csv(prepared_dir / "baseline_metrics.csv")
    assert sites.household_id.is_unique and set(sites.split) == {"train", "validation", "test"}
    assert sites.overlap_interval.sum() == 204
    eligible = sites[sites.eligible_2012_2014].copy()
    assert eligible.groupby("split").size().to_dict() == {"train": 4514, "validation": 974, "test": 969}
    if not include_test:
        eligible = eligible[eligible.split != "test"].copy()
    subset = panel[panel.household_id.isin(eligible.household_id) & panel.year.isin(YEARS)].copy()
    assert subset.groupby("household_id").size().eq(36).all()
    assert subset.usable_month.all()
    assert not subset.duplicated(["household_id", "month"]).any()

    lgas = sorted(eligible.loc[eligible.split == "train", "lga"].dropna().unique().tolist())
    lga_index = {name: i for i, name in enumerate(lgas)}
    result = {}
    for split in (("train", "validation", "test") if include_test else ("train", "validation")):
        selected = eligible[eligible.split == split].sort_values("household_id").reset_index(drop=True)
        block = subset[subset.household_id.isin(selected.household_id)].sort_values(["household_id", "month"])
        assert block.household_id.drop_duplicates().tolist() == selected.household_id.tolist()
        raw = block[list(CHANNELS)].to_numpy(np.float32).reshape(len(selected), 36, 3)
        assert np.isfinite(raw).all() and (raw >= 0).all()
        cond = np.zeros((len(selected), 2 + len(lgas) + 1), dtype=np.float32)
        cond[:, 0] = selected.cohort.eq("solar").astype(np.float32)
        cond[:, 1] = selected.capacity_kw.fillna(0).to_numpy(np.float32) / 10.0
        for row, lga in enumerate(selected.lga):
            cond[row, 2 + lga_index.get(lga, len(lgas))] = 1.0
        result[split] = {"sites": selected, "raw": raw, "cond": cond,
                         "solar": selected.cohort.eq("solar").to_numpy(),
                         "controlled_present": raw[:, :, 1] > 0.01}
    log_train = np.log1p(result["train"]["raw"]).reshape(-1, 3)
    mean = log_train.mean(axis=0).astype(np.float32)
    std = np.maximum(log_train.std(axis=0), 0.1).astype(np.float32)
    for data in result.values():
        data["x"] = ((np.log1p(data["raw"]) - mean) / std).astype(np.float32)
    scaler = {"mean": mean, "std": std, "lgas": lgas}
    return result, scaler, baseline


In [ ]:
class ConditionalVAE:
    """One-hidden-layer conditional VAE with an explicit controlled-load gate."""

    def __init__(self, condition_dim: int, seed: int, hidden: int = 128,
                 latent: int = 16, dtype=np.float32):
        rng = np.random.default_rng(seed)
        self.condition_dim, self.hidden, self.latent = condition_dim, hidden, latent
        self.dtype = dtype

        def weight(m, n):
            return (rng.standard_normal((m, n)) * np.sqrt(2.0 / (m + n))).astype(dtype)

        self.params = {
            "We": weight(108 + condition_dim, hidden), "be": np.zeros(hidden, dtype),
            "Wmu": weight(hidden, latent), "bmu": np.zeros(latent, dtype),
            "Wlv": weight(hidden, latent), "blv": np.zeros(latent, dtype),
            "Wd": weight(latent + condition_dim, hidden), "bd": np.zeros(hidden, dtype),
            "Wo": weight(hidden, 108 + 36), "bo": np.zeros(108 + 36, dtype),
        }

    def decode(self, z: np.ndarray, cond: np.ndarray, solar: np.ndarray | None = None):
        p = self.params
        h = np.tanh(np.concatenate([z, cond], axis=1) @ p["Wd"] + p["bd"])
        out = h @ p["Wo"] + p["bo"]
        return out[:, :108].reshape(-1, 36, 3), out[:, 108:], h

    def decoder_backward(self, z: np.ndarray, cond: np.ndarray, solar: np.ndarray,
                         hd: np.ndarray, dout: np.ndarray):
        p = self.params
        grad = {"Wo": hd.T @ dout, "bo": dout.sum(axis=0)}
        dhd = (dout @ p["Wo"].T) * (1.0 - hd ** 2)
        dec_in = np.concatenate([z, cond], axis=1)
        grad["Wd"] = dec_in.T @ dhd
        grad["bd"] = dhd.sum(axis=0)
        dz = (dhd @ p["Wd"].T)[:, :self.latent]
        return grad, dz

    def loss_and_grad(self, x: np.ndarray, cond: np.ndarray, solar: np.ndarray,
                      ctl: np.ndarray, beta: float, eps: np.ndarray):
        p = self.params
        b = x.shape[0]
        enc_input = np.concatenate([x.reshape(b, 108), cond], axis=1)
        he = np.tanh(enc_input @ p["We"] + p["be"])
        mu = he @ p["Wmu"] + p["bmu"]
        lv_raw = he @ p["Wlv"] + p["blv"]
        lv = np.clip(lv_raw, -6.0, 4.0)
        sd = np.exp(0.5 * lv)
        z = mu + sd * eps
        y, logits, hd = self.decode(z, cond, solar)
        diff = y - x
        ctl_f = ctl.astype(x.dtype)
        solar_f = solar.astype(x.dtype)[:, None]
        gate_weight = 0.3

        base = np.mean(diff[:, :, 0] ** 2)
        controlled = np.sum((diff[:, :, 1] ** 2) * ctl_f) / max(float(ctl_f.sum()), 1.0)
        generation = np.sum((diff[:, :, 2] ** 2) * solar_f) / max(float(solar_f.sum() * 36), 1.0)
        gate = np.mean(np.logaddexp(0.0, logits) - ctl_f * logits)
        kl = 0.5 * np.mean(np.sum(mu ** 2 + np.exp(lv) - 1.0 - lv, axis=1))
        total = float(base + controlled + generation + gate_weight * gate + beta * kl)

        dy = np.zeros_like(y)
        dy[:, :, 0] = 2.0 * diff[:, :, 0] / (b * 36)
        dy[:, :, 1] = 2.0 * diff[:, :, 1] * ctl_f / max(float(ctl_f.sum()), 1.0)
        dy[:, :, 2] = 2.0 * diff[:, :, 2] * solar_f / max(float(solar_f.sum() * 36), 1.0)
        sigmoid = 1.0 / (1.0 + np.exp(-np.clip(logits, -30, 30)))
        dlogits = gate_weight * (sigmoid - ctl_f) / (b * 36)
        dout = np.concatenate([dy.reshape(b, 108), dlogits], axis=1)

        grad, dz = self.decoder_backward(z, cond, solar, hd, dout)
        dmu = dz + beta * mu / b
        dlv = dz * (0.5 * sd * eps) + beta * 0.5 * (np.exp(lv) - 1.0) / b
        dlv *= ((lv_raw > -6.0) & (lv_raw < 4.0))
        grad["Wmu"] = he.T @ dmu
        grad["bmu"] = dmu.sum(axis=0)
        grad["Wlv"] = he.T @ dlv
        grad["blv"] = dlv.sum(axis=0)
        dhe = (dmu @ p["Wmu"].T + dlv @ p["Wlv"].T) * (1.0 - he ** 2)
        grad["We"] = enc_input.T @ dhe
        grad["be"] = dhe.sum(axis=0)
        return total, grad, {"base": float(base), "controlled": float(controlled),
                             "generation": float(generation), "gate": float(gate), "kl": float(kl)}


def train_one_seed(data: dict, seed: int, max_epochs: int = 120, patience: int = 18,
                   batch_size: int = 256, hidden: int = 128, latent: int = 16,
                   learning_rate: float = 0.001,
                   model_class: type[ConditionalVAE] = ConditionalVAE) -> tuple[ConditionalVAE, pd.DataFrame]:
    rng = np.random.default_rng(seed)
    train, val = data["train"], data["validation"]
    model = model_class(train["cond"].shape[1], seed, hidden, latent)
    m = {k: np.zeros_like(v) for k, v in model.params.items()}
    v = {k: np.zeros_like(w) for k, w in model.params.items()}
    step = 0
    best, bad, best_params, history = np.inf, 0, None, []
    for epoch in range(1, max_epochs + 1):
        beta = 0.03 * min(1.0, epoch / 20.0)
        training_losses = []
        for indices in np.array_split(rng.permutation(len(train["x"])),
                                      int(np.ceil(len(train["x"]) / batch_size))):
            eps = rng.standard_normal((len(indices), latent)).astype(np.float32)
            batch_loss, grad, _ = model.loss_and_grad(train["x"][indices], train["cond"][indices],
                                                       train["solar"][indices], train["controlled_present"][indices],
                                                       beta=beta, eps=eps)
            training_losses.append(batch_loss)
            norm = np.sqrt(sum(float(np.sum(g * g)) for g in grad.values()))
            if norm > 5:
                grad = {k: g * (5.0 / norm) for k, g in grad.items()}
            step += 1
            for key, g in grad.items():
                m[key] = 0.9 * m[key] + 0.1 * g
                v[key] = 0.999 * v[key] + 0.001 * (g * g)
                model.params[key] -= learning_rate * (m[key] / (1 - 0.9 ** step)) / (
                    np.sqrt(v[key] / (1 - 0.999 ** step)) + 1e-8)
        # Fixed validation noise makes early stopping comparisons stable.
        vrng = np.random.default_rng(9000 + seed)
        val_eps = vrng.standard_normal((len(val["x"]), latent)).astype(np.float32)
        val_loss, _, parts = model.loss_and_grad(val["x"], val["cond"], val["solar"],
                                                val["controlled_present"], beta=0.03, eps=val_eps)
        history.append({"seed": seed, "epoch": epoch, "training_loss": float(np.mean(training_losses)),
                        "validation_loss": val_loss, **parts})
        if np.isfinite(val_loss) and val_loss < best - 1e-4:
            best, bad = val_loss, 0
            best_params = {k: x.copy() for k, x in model.params.items()}
        else:
            bad += 1
        if epoch % 10 == 0 or epoch == 1 or bad >= patience:
            print(f"seed={seed} epoch={epoch} val_loss={val_loss:.4f} best={best:.4f}")
        if bad >= patience:
            break
    assert best_params is not None
    model.params = best_params
    return model, pd.DataFrame(history)


def sample_profiles(model: ConditionalVAE, data: dict, scaler: dict, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed + 100_000)
    sites = data["sites"]
    z = rng.standard_normal((len(sites), model.latent)).astype(np.float32)
    predicted, logits, _ = model.decode(z, data["cond"], data["solar"])
    log_kwh = predicted * scaler["std"][None, None, :] + scaler["mean"][None, None, :]
    values = np.expm1(np.clip(log_kwh, 0, np.log1p(50_000))).astype(np.float32)
    gate_prob = 1.0 / (1.0 + np.exp(-np.clip(logits, -30, 30)))
    values[:, :, 1] *= rng.random(gate_prob.shape) < gate_prob
    values[~data["solar"], :, 2] = 0.0
    months = pd.date_range("2012-01-01", "2014-12-01", freq="MS")
    capacity = sites.capacity_kw.fillna(0).to_numpy(np.float32)
    upper = capacity[:, None] * (months.days_in_month.to_numpy()[None, :] * 24)
    values[:, :, 2] = np.minimum(values[:, :, 2], upper)
    anonymous = rng.integers(0, 2**63, size=len(sites), dtype=np.int64)
    assert len(np.unique(anonymous)) == len(sites)
    frame = pd.DataFrame({
        "household_id": np.repeat(sites.household_id.to_numpy(), 36),
        "synthetic_id": np.repeat([f"vae_{seed}_{int(v):016x}" for v in anonymous], 36),
        "cohort": np.repeat(sites.cohort.to_numpy(), 36),
        "lga": np.repeat(sites.lga.to_numpy(), 36),
        "capacity_kw": np.repeat(sites.capacity_kw.to_numpy(), 36),
        "month": np.tile(months.to_numpy(), len(sites)),
        "base_kwh": values[:, :, 0].reshape(-1),
        "controlled_kwh": values[:, :, 1].reshape(-1),
        "generation_kwh": values[:, :, 2].reshape(-1),
    })
    frame["year"] = frame.month.dt.year
    frame["month_number"] = frame.month.dt.month
    frame["total_consumption_kwh"] = frame.base_kwh + frame.controlled_kwh
    # Shuffle whole trajectories so file order does not expose the sorted
    # test-household order used by the fixed split manifest.
    order = np.concatenate([np.arange(i * 36, (i + 1) * 36) for i in rng.permutation(len(sites))])
    return frame.iloc[order].reset_index(drop=True)


def marginal_metrics(real: pd.DataFrame, synthetic: pd.DataFrame, seed: int) -> pd.DataFrame:
    rows = []
    for cohort in ("solar", "non-solar"):
        r = real[real.cohort == cohort]
        s = synthetic[synthetic.cohort == cohort]
        for channel in ("total_consumption_kwh", "controlled_kwh", "generation_kwh"):
            if cohort == "non-solar" and channel == "generation_kwh":
                continue
            a = r.groupby(["household_id", "year"])[channel].sum().to_numpy()
            b = s.groupby(["household_id", "year"])[channel].sum().to_numpy()
            w1 = wasserstein_distance(a, b)
            med = np.median(a)
            month_w1 = np.mean([wasserstein_distance(r.loc[r.month_number == m, channel],
                                                       s.loc[s.month_number == m, channel]) for m in range(1, 13)])
            rows.append({"seed": seed, "cohort": cohort, "channel": channel,
                         "target_households": int(r.household_id.nunique()),
                         "annual_real_median_kwh": float(med),
                         "annual_synthetic_median_kwh": float(np.median(b)),
                         "annual_w1_kwh": float(w1),
                         "annual_w1_over_real_median": float(w1 / med) if med > 0 else np.nan,
                         "mean_calendar_month_w1_kwh": float(month_w1),
                         "zero_month_rate_real": float((r[channel] == 0).mean()),
                         "zero_month_rate_synthetic": float((s[channel] == 0).mean())})
    return pd.DataFrame(rows)


def relationship_metrics(real: pd.DataFrame, synthetic: pd.DataFrame, seed: int) -> pd.DataFrame:
    rows = []
    for name, df in (("real", real), ("synthetic", synthetic)):
        for cohort in ("solar", "non-solar"):
            annual = df[(df.cohort == cohort) & (df.year == 2014)].groupby("household_id").agg(
                consumption=("total_consumption_kwh", "sum"),
                controlled=("controlled_kwh", "sum"),
                generation=("generation_kwh", "sum"),
                capacity=("capacity_kw", "first"))
            rows.append({"seed": seed, "kind": name, "cohort": cohort, "households": len(annual),
                         "controlled_load_presence_fraction": float((annual.controlled > 0).mean()),
                         "capacity_generation_spearman": float(spearmanr(annual.capacity, annual.generation).statistic) if cohort == "solar" else np.nan,
                         "consumption_generation_spearman": float(spearmanr(annual.consumption, annual.generation).statistic) if cohort == "solar" else np.nan})
    return pd.DataFrame(rows)


In [ ]:
def validation_halves(validation: dict, seed: int = 20260928) -> dict:
    rng = np.random.default_rng(seed)
    parts = {"calibration": [], "evaluation": []}
    for cohort in (True, False):
        idx = rng.permutation(np.flatnonzero(validation["solar"] == cohort))
        midpoint = len(idx) // 2
        parts["calibration"].extend(idx[:midpoint])
        parts["evaluation"].extend(idx[midpoint:])

    def subset(indices):
        indices = np.array(sorted(indices), dtype=int)
        return {key: (value.iloc[indices].reset_index(drop=True) if key == "sites"
                      else value[indices]) for key, value in validation.items()}

    result = {name: subset(indices) for name, indices in parts.items()}
    assert set(result["calibration"]["sites"].household_id).isdisjoint(result["evaluation"]["sites"].household_id)
    return result


def load_shared_models(checkpoint_dir: Path, data: dict, scaler: dict,
                       panel_hash: str, splits_hash: str) -> dict:
    config_path = checkpoint_dir / "config.json"
    if not config_path.is_file():
        raise FileNotFoundError("Run Ausgrid_Monthly_Conditional_VAE.ipynb first; config.json is missing.")
    config = json.loads(config_path.read_text())
    assert config["input_panel_sha256"] == panel_hash and config["input_splits_sha256"] == splits_hash, "Prepared inputs changed since Phase 2."
    assert config["training_lga_vocabulary"] == scaler["lgas"]
    assert np.allclose(config["scaler_mean_log1p_kwh"], scaler["mean"])
    assert np.allclose(config["scaler_std_log1p_kwh"], scaler["std"])
    assert tuple(config["seeds"]) == SEEDS
    models = {}
    for seed in SEEDS:
        path = checkpoint_dir / f"conditional_vae_seed{seed}.npz"
        if not path.is_file():
            raise FileNotFoundError(f"Run Phase 2 seed {seed}; missing {path}")
        model = ConditionalVAE(data["train"]["cond"].shape[1], seed,
                               hidden=config["hidden"], latent=config["latent"])
        with np.load(path) as archive:
            assert set(archive.files) == set(model.params)
            for key in model.params:
                assert archive[key].shape == model.params[key].shape
                model.params[key] = archive[key]
        models[seed] = model
    return models


def real_validation_panel(panel_path: Path, val_ids: set[str]) -> pd.DataFrame:
    df = pd.read_csv(panel_path, parse_dates=["month"])
    return df[df.household_id.isin(val_ids) & df.year.isin(YEARS)].copy()


In [ ]:
PROJECT_ROOT = resolve_project_root()
PREPARED = PROJECT_ROOT / "monthly_work_package_1"
PHASE2 = PROJECT_ROOT / "outputs" / "monthly_vae"
OUTPUT = PROJECT_ROOT / "outputs" / "monthly_vae_ablation"
OUTPUT.mkdir(parents=True, exist_ok=True)
DATA, SCALER, _ = load_monthly_data(PREPARED, include_test=False)
PARTS = validation_halves(DATA["validation"])
PANEL_PATH = PREPARED / "monthly_panel_2007_2014.csv.gz"
SPLITS_PATH = PREPARED / "household_splits.csv"
SHARED = load_shared_models(PHASE2, DATA, SCALER,
                            sha256_file(PANEL_PATH), sha256_file(SPLITS_PATH))
cal_ids = set(PARTS["calibration"]["sites"].household_id)
eval_ids = set(PARTS["evaluation"]["sites"].household_id)
assert cal_ids.isdisjoint(eval_ids)
REAL_CAL = real_validation_panel(PANEL_PATH, cal_ids)
REAL_EVAL = real_validation_panel(PANEL_PATH, eval_ids)
assert len(REAL_CAL) == 487 * 36 and len(REAL_EVAL) == 487 * 36
print("Project:", PROJECT_ROOT)
print("Training:", len(DATA["train"]["sites"]), "| calibration:", len(cal_ids), "| evaluation:", len(eval_ids))
print("Prior test trajectories loaded: no")


## 2. Test and train cohort-specific decoder heads

The shared encoder, latent dimension, reconstruction loss, KL weight, optimizer, training households, seeds, and early-stopping rule are unchanged. Each decoder head starts from the same initialization as the original single decoder. The gradient check tests both heads separately. This is one architecture change; compare it against the loaded single-decoder checkpoints.


In [ ]:
class CohortDecoderVAE(ConditionalVAE):
    """Same encoder and loss; separate decoder parameters for each cohort."""

    def __init__(self, condition_dim: int, seed: int, hidden: int = 128,
                 latent: int = 16, dtype=np.float32):
        super().__init__(condition_dim, seed, hidden, latent, dtype)
        for suffix in ("solar", "non_solar"):
            for key in ("Wd", "bd", "Wo", "bo"):
                self.params[f"{key}_{suffix}"] = self.params[key].copy()
        for key in ("Wd", "bd", "Wo", "bo"):
            del self.params[key]

    def decode(self, z: np.ndarray, cond: np.ndarray, solar: np.ndarray | None = None):
        if solar is None:
            raise ValueError("The cohort-specific decoder requires a solar mask.")
        p = self.params
        dec_input = np.concatenate([z, cond], axis=1)
        hidden = np.empty((len(z), self.hidden), dtype=z.dtype)
        out = np.empty((len(z), 144), dtype=z.dtype)
        for mask, suffix in ((solar, "solar"), (~solar, "non_solar")):
            if not np.any(mask):
                continue
            h = np.tanh(dec_input[mask] @ p[f"Wd_{suffix}"] + p[f"bd_{suffix}"])
            hidden[mask] = h
            out[mask] = h @ p[f"Wo_{suffix}"] + p[f"bo_{suffix}"]
        return out[:, :108].reshape(-1, 36, 3), out[:, 108:], hidden

    def decoder_backward(self, z: np.ndarray, cond: np.ndarray, solar: np.ndarray,
                         hd: np.ndarray, dout: np.ndarray):
        p = self.params
        dec_input = np.concatenate([z, cond], axis=1)
        dz = np.zeros_like(z)
        grad = {}
        for mask, suffix in ((solar, "solar"), (~solar, "non_solar")):
            grad[f"Wd_{suffix}"] = np.zeros_like(p[f"Wd_{suffix}"])
            grad[f"bd_{suffix}"] = np.zeros_like(p[f"bd_{suffix}"])
            grad[f"Wo_{suffix}"] = np.zeros_like(p[f"Wo_{suffix}"])
            grad[f"bo_{suffix}"] = np.zeros_like(p[f"bo_{suffix}"])
            if not np.any(mask):
                continue
            d = dout[mask]
            h = hd[mask]
            grad[f"Wo_{suffix}"] = h.T @ d
            grad[f"bo_{suffix}"] = d.sum(axis=0)
            dh = (d @ p[f"Wo_{suffix}"].T) * (1.0 - h ** 2)
            grad[f"Wd_{suffix}"] = dec_input[mask].T @ dh
            grad[f"bd_{suffix}"] = dh.sum(axis=0)
            dz[mask] = (dh @ p[f"Wd_{suffix}"].T)[:, :self.latent]
        return grad, dz


def check_cohort_gradients() -> dict:
    rng = np.random.default_rng(212)
    model = CohortDecoderVAE(4, seed=11, hidden=7, latent=3, dtype=np.float64)
    x = rng.normal(size=(5, 36, 3))
    cond = rng.normal(size=(5, 4))
    solar = np.array([True, False, True, False, True])
    ctl = rng.random((5, 36)) < 0.35
    eps = rng.normal(size=(5, 3))
    _, grad, _ = model.loss_and_grad(x, cond, solar, ctl, beta=0.03, eps=eps)
    checked = {}
    for key, index in (("We", (0, 0)), ("Wd_solar", (0, 0)),
                       ("Wo_solar", (1, 4)), ("Wd_non_solar", (0, 1)),
                       ("Wo_non_solar", (1, 111)), ("bo_non_solar", (109,))):
        p = model.params[key]
        old = float(p[index])
        delta = 1e-5
        p[index] = old + delta
        hi = model.loss_and_grad(x, cond, solar, ctl, beta=0.03, eps=eps)[0]
        p[index] = old - delta
        lo = model.loss_and_grad(x, cond, solar, ctl, beta=0.03, eps=eps)[0]
        p[index] = old
        numeric = (hi - lo) / (2 * delta)
        analytic = float(grad[key][index])
        error = abs(numeric - analytic) / max(1e-5, abs(numeric), abs(analytic))
        checked[f"{key}_{index}"] = error
        assert error < 5e-4, (key, analytic, numeric)
    return checked


In [ ]:
errors = check_cohort_gradients()
print("Two-decoder gradient check passed; largest relative error:", max(errors.values()))
MODELS, HISTORIES, DURATIONS = {}, [], {}
for seed in SEEDS:
    start = time.perf_counter()
    model, history = train_one_seed(
        DATA, seed, max_epochs=120, patience=18, batch_size=256,
        hidden=128, latent=16, learning_rate=0.001,
        model_class=CohortDecoderVAE)
    MODELS[seed] = model
    HISTORIES.append(history)
    DURATIONS[str(seed)] = round(time.perf_counter() - start, 1)
    np.savez_compressed(OUTPUT / f"two_decoder_seed{seed}.npz", **model.params)
    print(f"seed={seed}, best validation epoch={int(history.loc[history.validation_loss.idxmin(), 'epoch'])}")
pd.concat(HISTORIES, ignore_index=True).to_csv(OUTPUT / "training_history.csv", index=False)


## 3. Compare on the *evaluation half* of validation

Regenerate the first notebook's training-only seasonal-resampling baseline for validation; its saved synthetic CSV contains **test** targets and is therefore unsuitable here. For each seed, sample shared and two-decoder VAEs on both validation halves. Fit correction factors using **only the calibration half** and score on the other half. Keep controlled load uncalibrated because its median annual value is zero; inspect its absolute kWh W1 and zero rate.


In [ ]:
def resample_validation(train: dict, validation: dict, seed: int) -> pd.DataFrame:
    """Re-run the first work package's training-only seasonal baseline on validation."""
    rng = np.random.default_rng(seed)
    donors, targets = train["sites"], validation["sites"]
    rows = []
    months = pd.date_range("2012-01-01", "2014-12-01", freq="MS")
    for i, target in enumerate(targets.itertuples(index=False)):
        candidates = donors[donors.cohort == target.cohort]
        in_lga = candidates.lga.eq(target.lga)
        if target.cohort == "solar":
            near = (candidates.capacity_kw - target.capacity_kw).abs() <= 0.75
            choices = [in_lga & near, near, in_lga]
        else:
            choices = [in_lga]
        selected = candidates
        for mask in choices:
            if int(mask.sum()) >= 10:
                selected = candidates[mask]
                break
        donor = selected.iloc[rng.integers(len(selected))]
        donor_idx = donors.index.get_loc(donor.name)
        values = train["raw"][donor_idx].copy()
        cons_noise = rng.lognormal(mean=-0.5 * 0.08**2, sigma=0.08, size=3)
        gen_noise = rng.lognormal(mean=-0.5 * 0.08**2, sigma=0.08, size=3)
        for year in range(3):
            values[year * 12:(year + 1) * 12, :2] *= cons_noise[year]
            if target.cohort == "solar":
                values[year * 12:(year + 1) * 12, 2] *= (target.capacity_kw / donor.capacity_kw) * gen_noise[year]
        rows.append(pd.DataFrame({
            "household_id": target.household_id, "cohort": target.cohort,
            "capacity_kw": target.capacity_kw, "month": months,
            "base_kwh": values[:, 0], "controlled_kwh": values[:, 1],
            "generation_kwh": values[:, 2],
        }))
    result = pd.concat(rows, ignore_index=True)
    result["year"] = result.month.dt.year
    result["month_number"] = result.month.dt.month
    result["total_consumption_kwh"] = result.base_kwh + result.controlled_kwh
    return result


def fit_annual_factors(real_cal: pd.DataFrame, synth_cal: pd.DataFrame) -> dict:
    """Fit median annual kWh ratios on the calibration half only."""
    result = {}
    for cohort in ("solar", "non-solar"):
        for channel in (("base_kwh", "generation_kwh") if cohort == "solar" else ("base_kwh",)):
            real_year = real_cal[real_cal.cohort == cohort].groupby(["household_id", "year"])[channel].sum()
            syn_year = synth_cal[synth_cal.cohort == cohort].groupby(["household_id", "year"])[channel].sum()
            denom = float(np.median(syn_year))
            factor = float(np.clip(np.median(real_year) / max(denom, 1e-8), 0.5, 2.0))
            result[f"{cohort}:{channel}"] = factor
    return result


def apply_annual_factors(synthetic: pd.DataFrame, factors: dict) -> pd.DataFrame:
    result = synthetic.copy()
    for label, factor in factors.items():
        cohort, channel = label.split(":")
        result.loc[result.cohort == cohort, channel] *= factor
    upper = result.capacity_kw.fillna(0) * 24 * result.month.dt.days_in_month
    result["generation_kwh"] = np.minimum(result.generation_kwh, upper)
    result["total_consumption_kwh"] = result.base_kwh + result.controlled_kwh
    return result


def diagnostic_quantiles(real: pd.DataFrame, candidates: dict[str, pd.DataFrame],
                         train_capacity: np.ndarray) -> pd.DataFrame:
    """Annual tail and yield summaries without any test-household records."""
    rows = []
    cuts = np.quantile(train_capacity, [1 / 3, 2 / 3])
    for method, frame in {"real": real, **candidates}.items():
        for cohort in ("solar", "non-solar"):
            block = frame[frame.cohort == cohort]
            annual = block.groupby(["household_id", "year"]).agg(
                base_kwh=("base_kwh", "sum"), controlled_kwh=("controlled_kwh", "sum"),
                generation_kwh=("generation_kwh", "sum"),
                total_consumption_kwh=("total_consumption_kwh", "sum"),
                capacity_kw=("capacity_kw", "first"))
            annual["capacity_band"] = (np.searchsorted(cuts, annual.capacity_kw.fillna(0), side="right")
                                       if cohort == "solar" else -1)
            annual["yield_kwh_per_kw"] = (annual.generation_kwh / annual.capacity_kw
                                          if cohort == "solar" else np.nan)
            for band, grp in annual.groupby("capacity_band"):
                for channel in ("total_consumption_kwh", "controlled_kwh", "generation_kwh", "yield_kwh_per_kw"):
                    if cohort == "non-solar" and channel in ("generation_kwh", "yield_kwh_per_kw"):
                        continue
                    values = grp[channel].dropna().to_numpy()
                    q = np.quantile(values, [.05, .25, .5, .75, .95])
                    rows.append({"method": method, "cohort": cohort, "capacity_band": int(band),
                                 "channel": channel, "n_household_years": len(values),
                                 **{f"q{p:02d}": float(v) for p, v in zip((5, 25, 50, 75, 95), q)}})
    return pd.DataFrame(rows)


def validation_membership_probe(model: ConditionalVAE, data: dict, eval_subset: dict,
                                scaler: dict, seed: int) -> pd.DataFrame:
    """Exploratory train-vs-evaluation-validation nearest-neighbor attack."""
    rng = np.random.default_rng(seed + 50_000)
    rows = []
    for cohort in ("solar", "non-solar"):
        solar = cohort == "solar"
        train_idx = np.flatnonzero(data["train"]["solar"] == solar)
        eval_idx = np.flatnonzero(eval_subset["solar"] == solar)
        chosen = rng.choice(train_idx, len(eval_idx), replace=False)
        sampled = {key: (value.iloc[chosen].reset_index(drop=True) if key == "sites" else value[chosen])
                   for key, value in data["train"].items()}
        pool = sample_profiles(model, sampled, scaler, seed + (1 if solar else 2))
        pool_raw = pool[list(CHANNELS)].to_numpy(np.float32).reshape(len(eval_idx), 36, 3)
        channels = (0, 1, 2) if solar else (0, 1)

        def features(raw):
            z = (np.log1p(raw) - scaler["mean"]) / scaler["std"]
            return z[:, :, channels].reshape(len(raw), -1).astype(np.float64)

        pool_features = features(pool_raw)
        train_dist = cdist(features(data["train"]["raw"][chosen]), pool_features).min(axis=1)
        eval_dist = cdist(features(eval_subset["raw"][eval_idx]), pool_features).min(axis=1)
        auc = roc_auc_score(np.r_[np.ones(len(eval_idx)), np.zeros(len(eval_idx))],
                            -np.r_[train_dist, eval_dist])
        rows.append({"seed": seed, "cohort": cohort, "member_train": len(eval_idx),
                     "nonmember_validation_eval": len(eval_idx),
                     "attack_auc": float(auc), "train_median_distance": float(np.median(train_dist)),
                     "validation_median_distance": float(np.median(eval_dist))})
    return pd.DataFrame(rows)


def plot_validation(real: pd.DataFrame, candidates: dict[str, pd.DataFrame], path: Path):
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), constrained_layout=True)
    cases = (("solar", "total_consumption_kwh", "Solar consumption"),
             ("solar", "generation_kwh", "Solar generation"),
             ("non-solar", "total_consumption_kwh", "Non-solar consumption"))
    for ax, (cohort, channel, title) in zip(axes, cases):
        for label, df, style in [("Real", real, "-"),
                                 ("Resampling", candidates["resampling"], ":"),
                                 ("Shared decoder", candidates["shared"], "--"),
                                 ("Two decoders", candidates["two_decoders"], "-.")]:
            monthly = df[(df.cohort == cohort) & (df.year == 2014)].groupby("month_number")[channel].median()
            ax.plot(monthly.index, monthly.to_numpy(), style, linewidth=1.8, label=label)
        ax.set(title=title, xlabel="Month of 2014", ylabel="Median kWh per household")
        ax.set_xticks([1, 3, 5, 7, 9, 11])
        ax.grid(alpha=0.2)
    axes[0].legend(frameon=False)
    fig.savefig(path, dpi=170)
    plt.close(fig)


In [ ]:
METRICS, RELATIONS, PRIVACY, QUANTILES, FACTORS = [], [], [], [], []
train_capacity = DATA["train"]["sites"].loc[DATA["train"]["solar"], "capacity_kw"].to_numpy()
for seed in SEEDS:
    baseline = resample_validation(DATA["train"], PARTS["evaluation"], seed)
    shared_eval = sample_profiles(SHARED[seed], PARTS["evaluation"], SCALER, seed)
    two_eval = sample_profiles(MODELS[seed], PARTS["evaluation"], SCALER, seed)
    two_cal = sample_profiles(MODELS[seed], PARTS["calibration"], SCALER, seed)
    factors = fit_annual_factors(REAL_CAL, two_cal)
    calibrated = apply_annual_factors(two_eval, factors)
    FACTORS.append({"seed": seed, **factors})
    candidates = {"resampling": baseline, "shared": shared_eval,
                  "two_decoders": two_eval, "two_decoders_calibrated": calibrated}
    for method, generated in candidates.items():
        assert set(generated.household_id.unique()) == eval_ids
        assert generated[list(CHANNELS)].ge(0).all().all()
        score = marginal_metrics(REAL_EVAL, generated, seed)
        score["method"] = method
        METRICS.append(score)
        relationships = relationship_metrics(REAL_EVAL, generated, seed)
        RELATIONS.append(relationships[relationships.kind == "synthetic"].assign(method=method))
    QUANTILES.append(diagnostic_quantiles(REAL_EVAL, candidates, train_capacity).assign(seed=seed))
    for label, model in (("shared", SHARED[seed]), ("two_decoders", MODELS[seed])):
        PRIVACY.append(validation_membership_probe(model, DATA, PARTS["evaluation"], SCALER, seed).assign(method=label))
    if seed == SEEDS[0]:
        plot_validation(REAL_EVAL, candidates, OUTPUT / "validation_seasonality_2014.png")
        two_eval.drop(columns="household_id").to_csv(
            OUTPUT / "two_decoder_validation_eval_seed11.csv.gz", index=False, compression="gzip")

metrics = pd.concat(METRICS, ignore_index=True)
relations = pd.concat(RELATIONS, ignore_index=True)
privacy = pd.concat(PRIVACY, ignore_index=True)
quantiles = pd.concat(QUANTILES, ignore_index=True)
factors = pd.DataFrame(FACTORS)
metrics.to_csv(OUTPUT / "validation_metrics.csv", index=False)
relations.to_csv(OUTPUT / "validation_relationship_metrics.csv", index=False)
privacy.to_csv(OUTPUT / "validation_membership_probe.csv", index=False)
quantiles.to_csv(OUTPUT / "validation_annual_quantiles.csv", index=False)
factors.to_csv(OUTPUT / "calibration_factors.csv", index=False)

comparison = metrics.groupby(["method", "cohort", "channel"])[
    ["annual_w1_kwh", "annual_w1_over_real_median", "mean_calendar_month_w1_kwh",
     "zero_month_rate_synthetic"]].mean().reset_index()
comparison.to_csv(OUTPUT / "validation_comparison.csv", index=False)
record = {"input_panel_sha256": sha256_file(PANEL_PATH),
          "input_splits_sha256": sha256_file(SPLITS_PATH),
          "validation_calibration_ids": len(cal_ids), "validation_evaluation_ids": len(eval_ids),
          "training_seconds": DURATIONS,
          "protocol": "Only train and validation trajectories; test data not loaded. Calibration factors fitted on validation calibration half; comparisons on validation evaluation half. Shared checkpoints used whole validation for early stopping.",
          "privacy_note": "Exploratory single nearest-neighbor membership probe; AUC near 0.5 is no release guarantee."}
(OUTPUT / "experiment_manifest.json").write_text(json.dumps(record, indent=2) + "\n")
print(comparison[["method", "cohort", "channel", "annual_w1_over_real_median"]].round(3).to_string(index=False))
print()
print("Privacy AUC means:", privacy.groupby(["method", "cohort"]).attack_auc.mean().round(3))
print()
print("Validation calibration factors:", factors.round(3).to_string(index=False))


## Interpretation

Read `validation_annual_quantiles.csv` alongside W1: a visually accurate median solar profile can still miss household-level spread and tails. `validation_comparison.csv` includes **absolute annual kWh W1** for controlled load, whose normalized W1 is undefined when the real annual median equals zero.

Choose any further architecture or calibration work using this validation experiment. Because the earlier 2012–2014 test set was already examined, repeating scores there later is exploratory. For a final unbiased research comparison, lock the method and create a new household-held-out split, then retrain all models before evaluating that new holdout once. The 2011–2013 interval-overlap experiment remains separate.


In [ ]:
from IPython.display import display, Image
display(Image(filename=str(OUTPUT / "validation_seasonality_2014.png")))
print("Saved to:", OUTPUT)
print("Files:", [p.name for p in sorted(OUTPUT.iterdir())])
